# FEATURE EXTRACTION

Note: The following contains the code used for DINOv2 feature extraction from raw images. The full, raw image datasets used in the experiment are not uploaded due to the large file sizes (20+ GB). However, the sources of the image datasets are provided below. This code is only provided for reference: you need to provide your own image paths if you wish to run it.

## Image Download Links:

Links to imagesets used:

Training the Rectified Flow models:
- ImageNet: https://huggingface.co/datasets/ILSVRC/imagenet-1k
- LSUN Bedrooms: https://huggingface.co/datasets/pcuenq/lsun-bedrooms
- CelebA HQ: https://huggingface.co/datasets/korexyz/celeba-hq-256x256

CNN Training:
- ImageNet: https://huggingface.co/datasets/ILSVRC/imagenet-1k
- FFHQ: https://huggingface.co/datasets/marcosv/ffhq-dataset
- LSUN Bedrooms: https://huggingface.co/datasets/pcuenq/lsun-bedrooms
- GenImage: https://github.com/GenImage-Dataset/GenImage

Benchmarks:
- ImageNet: https://huggingface.co/datasets/ILSVRC/imagenet-1k
- DALL-E 3: https://huggingface.co/datasets/ehristoforu/dalle-3-images
- FLUX: https://civitai.com/models/631007/dataset-with-6000-flux1-dev-images-1024x768-and-768x1024?modelVersionId=705402
- Midjourney V6: https://huggingface.co/datasets/brivangl/midjourney-v6-llava
- Nano Banana: https://huggingface.co/datasets/bitmind/nano-banana
- Stable Diffusion 3: https://huggingface.co/datasets/GeroldMeisinger/laion2b-en-a65_cogvlm2-4bit_captions
- WildRF: https://github.com/barcavia/RealTime-DeepfakeDetection-in-the-RealWorld

In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/FLOWTRACER')
print("Current Working Directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory: /content/drive/MyDrive/FLOWTRACER


## DINOv2 Extraction

In [ ]:
import os
import torch
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

# Insert path containing the raw images
image_dir = ''
save_dir = ''
os.makedirs(save_dir, exist_ok=True)

# DINOv2 Setup
print("Loading DINOv2 ViT-L/14 (Large)...")
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').cuda().half().eval()

# Necessary DINOv2 preprocessing
train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),         
    transforms.CenterCrop(224),     
    transforms.ToTensor(),          
    transforms.Normalize(           
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

# Extraction
print("\nStarting Extraction...")

# These are the number of images used in the experiment.
dataset_limits = {
    "imgnet": 50000,
    "celeba": 30000,
    "lsun": 50000
}

BATCH_SIZE = 64 

for prefix, limit in dataset_limits.items():
    prefix_dir = os.path.join(image_dir, prefix)
    print(f"\n Currently processing: {prefix.upper()}")
    
    batch_imgs = []
    batch_paths = []
    
    for count in tqdm(range(limit), desc=f"Extracting {prefix}"):
        filename = f"{count:07d}.jpg"
        img_path = os.path.join(prefix_dir, filename)
        
        if not os.path.exists(img_path):
            continue
            
        feature_filename = f"{prefix}_{count:07d}.pt"
        save_path = os.path.join(save_dir, feature_filename)
        
        if os.path.exists(save_path):
            continue
            
        try:
            raw_img = Image.open(img_path).convert('RGB')
            img_t = train_transform(raw_img)
            
            batch_imgs.append(img_t)
            batch_paths.append(save_path)
            
            if len(batch_imgs) == BATCH_SIZE:
                batch_tensor = torch.stack(batch_imgs).cuda().half()
                
                with torch.inference_mode():
                    features = model.forward_features(batch_tensor)
                    patch_tokens = features['x_norm_patchtokens']
                    
                    mean_feat = patch_tokens.mean(dim=1)
                    std_feat = patch_tokens.std(dim=1)
                    combined_feat = torch.cat([mean_feat, std_feat], dim=1)
                    normalized_feat = torch.nn.functional.normalize(combined_feat, p=2, dim=1).cpu()
                
                for i in range(len(batch_paths)):
                    tiny_tensor = normalized_feat[i].unsqueeze(0).clone()
                    torch.save(tiny_tensor, batch_paths[i])
                    
                batch_imgs = []
                batch_paths = []
                
        except Exception as e:
            print(f"\n CRASH ON {filename}: {e}")
            
    # Process leftover images
    if len(batch_imgs) > 0:
        batch_tensor = torch.stack(batch_imgs).cuda().half()
        with torch.inference_mode():
            features = model.forward_features(batch_tensor)
            patch_tokens = features['x_norm_patchtokens']
            mean_feat = patch_tokens.mean(dim=1)
            std_feat = patch_tokens.std(dim=1)
            combined_feat = torch.cat([mean_feat, std_feat], dim=1)
            normalized_feat = torch.nn.functional.normalize(combined_feat, p=2, dim=1).cpu()
        
        for i in range(len(batch_paths)):
            tiny_tensor = normalized_feat[i].unsqueeze(0).clone()
            torch.save(tiny_tensor, batch_paths[i])

print(f"\n Features saved in {save_dir}!")